In [1]:
import os
import pandas as pd
from dotenv import load_dotenv

from langfuse import Langfuse
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(override=True)

langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_LOCAL_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_LOCAL_SECRET_KEY"),
    base_url=os.getenv("LANGFUSE_LOCAL_BASE_URL")
)

print("Langfuse:", langfuse.auth_check())

Langfuse: True


In [3]:
from pathlib import Path

for path in [
    "G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_001",
    "G:\\SEVA_Ai\\Seva_ai_data\\vs_data\\vectorstores\\chunking_comparison\\exp_002_v2",
    "G:/SEVA_Ai/SEVA_Ai/notebooks/golden_dataset/final"
]:
    print(path, "->", Path(path).exists())

G:\SEVA_Ai\Seva_ai_data\vs_data\vectorstores\chunking_comparison\exp_001 -> True
G:\SEVA_Ai\Seva_ai_data\vs_data\vectorstores\chunking_comparison\exp_002_v2 -> True
G:/SEVA_Ai/SEVA_Ai/notebooks/golden_dataset/final -> True


In [4]:
# 1. Initialize the embedding model
# Use exactly the same embedding model and configuration used during indexing.
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 16}
)

g:\SEVA_Ai\SEVA_Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 457.60it/s]


In [5]:
#. Load both Chroma collections
from langchain_chroma import Chroma

exp_001 = Chroma(
    collection_name="exp_001_recursive",
    persist_directory=r"G:\SEVA_Ai\Seva_ai_data\vs_data\vectorstores\chunking_comparison\exp_001",
    embedding_function=embedding_model
)

exp_002 = Chroma(
    collection_name="exp_002_structure_v2",
    persist_directory=r"G:\SEVA_Ai\Seva_ai_data\vs_data\vectorstores\chunking_comparison\exp_002_v2",
    embedding_function=embedding_model
)

In [6]:
#3. Verify collection counts
print("EXP_001:", exp_001._collection.count())
print("EXP_002_v2:", exp_002._collection.count())

EXP_001: 124029
EXP_002_v2: 91858


In [7]:
#4. Test retrieval
query = "What financial assistance is available for farmers?"

results_001 = exp_001.similarity_search_with_score(query, k=5)
results_002 = exp_002.similarity_search_with_score(query, k=5)

print("EXP_001 results:", len(results_001))
print("EXP_002 results:", len(results_002))

EXP_001 results: 5
EXP_002 results: 5


In [10]:
import json
path = Path("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits\\test.json")
with open(path, "r", encoding="utf-8") as f:
    test_records = json.load(f)

for row in test_records:
    langfuse.create_dataset_item(
        dataset_name="seva-ai-retrieval-golden-v1",

        input={
            "question": row["question"]
        },

        expected_output={
            "relevant_scheme_ids": row["relevant_scheme_ids"],
            "relevant_doc_ids": row["relevant_doc_ids"]
        },

        metadata={
            "test_id": row["test_id"],
            "faq_id": row["faq_id"],
            "scheme_id": row["scheme_id"],
            "scheme_name": row["scheme_name"],
            "categories": row["categories"],
            "level": row["level"],
            "answerable": row["answerable"]
        }
    )

langfuse.flush()

print("Dataset upload completed!")

Dataset upload completed!


In [11]:
dataset = langfuse.get_dataset("seva-ai-retrieval-golden-v1")
print("Total items:", len(dataset.items))

Total items: 973


In [12]:
# Fetch the Langfuse dataset
dataset = langfuse.get_dataset("seva-ai-retrieval-golden-v1")

# Convert Langfuse items into a simple evaluation format
benchmark = []

for item in dataset.items:
    benchmark.append({
        "item_id": item.id,
        "test_id": item.metadata["test_id"],
        "question": item.input["question"],
        "relevant_scheme_ids": item.expected_output["relevant_scheme_ids"]
    })

# Validation
assert len(benchmark) == 973
assert len({x["test_id"] for x in benchmark}) == 973
assert all(x["relevant_scheme_ids"] for x in benchmark)

print("Total questions:", len(benchmark))
print("Unique test IDs:", len({x["test_id"] for x in benchmark}))
print("\nSample:")
print(benchmark[0])

Total questions: 973
Unique test IDs: 973

Sample:
{'item_id': 'b7264109-7c23-429b-a1a4-c264b5ea7e5e', 'test_id': 'GD_000009', 'question': 'Which documents are required ?', 'relevant_scheme_ids': ['06H9F5gBXu-iUKXqbHjX']}


In [13]:
import langfuse

print("Langfuse SDK version:", langfuse.__version__)

Langfuse SDK version: 5.2.0


In [ ]:
def retrieve_documents(vector_store, query, k=100):
    results = vector_store.similarity_search_with_score(query, k=k)

    retrieved = []

    for rank, (doc, distance) in enumerate(results, start=1):
        retrieved.append({
            "rank": rank,
            "chunk_id": doc.id,
            "scheme_id": doc.metadata.get("scheme_id"),
            "parent_id": doc.metadata.get("parent_id"),
            "section": doc.metadata.get("section"),
            "text": doc.page_content,
            "distance": float(distance)
        })

    return retrieved

In [15]:
query = benchmark[0]["question"]

results_001 = retrieve_documents(exp_001, query, k=100)
results_002 = retrieve_documents(exp_002, query, k=100)

print("Question:", query)
print("EXP_001 chunks:", len(results_001))
print("EXP_002 chunks:", len(results_002))

print("\nEXP_001 first result:")
print(results_001[0])

print("\nEXP_002 first result:")
print(results_002[0])

Question: Which documents are required ?
EXP_001 chunks: 100
EXP_002 chunks: 100

EXP_001 first result:
{'rank': 1, 'chunk_id': None, 'scheme_id': 'jsbCF5gB0AGc0F0iCTY0', 'section': 'faq', 'text': 'Question: What documents are mandatory?\nAnswer: Identity, caste, training, DPR/SCP, photos, bills, and bank details.', 'distance': 0.34520599246025085}

EXP_002 first result:
{'rank': 1, 'chunk_id': None, 'scheme_id': 'EcbXF5gB0AGc0F0i4jfD', 'section': 'documents_required', 'text': 'Any other documents as required', 'distance': 0.21955834329128265}


In [16]:
for name, results in [
    ("EXP_001", results_001),
    ("EXP_002_v2", results_002)
]:
    unique_schemes = {
        r["scheme_id"] for r in results
        if r["scheme_id"] is not None
    }

    print(name, "Unique schemes:", len(unique_schemes))

EXP_001 Unique schemes: 97
EXP_002_v2 Unique schemes: 100


In [17]:
print("EXP_001 metadata:")
print(results_001[0])

print("\nEXP_002 metadata:")
print(results_002[0])

EXP_001 metadata:
{'rank': 1, 'chunk_id': None, 'scheme_id': 'jsbCF5gB0AGc0F0iCTY0', 'section': 'faq', 'text': 'Question: What documents are mandatory?\nAnswer: Identity, caste, training, DPR/SCP, photos, bills, and bank details.', 'distance': 0.34520599246025085}

EXP_002 metadata:
{'rank': 1, 'chunk_id': None, 'scheme_id': 'EcbXF5gB0AGc0F0i4jfD', 'section': 'documents_required', 'text': 'Any other documents as required', 'distance': 0.21955834329128265}


In [18]:
print("EXP_001 keys:", exp_001._collection.get(limit=1)["metadatas"][0].keys())

print("EXP_002 keys:", exp_002._collection.get(limit=1)["metadatas"][0].keys())

EXP_001 keys: dict_keys(['categories', 'section', 'document_type', 'experiment', 'parent_id', 'scheme_id'])
EXP_002 keys: dict_keys(['scheme_id', 'experiment', 'parent_id', 'section', 'document_type', 'categories'])
